# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ifahadkareem/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

### 1. What one row means

In the raw warehouse table, one row represents one content item for one client on one report date. For modeling, I aggregate these daily rows into one row per client-content pair at the March 25, 2026 decision moment.

### 2. Which table I use

I use `fact_content_daily_performance`, specifically the `month=2026-03` partition.

I do not use `fact_content_daily_performance_sample` because that table contains June 2026, the final sealed month.

### 3. Which time window I use

My feature window is March 1–24, 2026.

My recent baseline window is March 18–24, 2026.

My future outcome window is March 25–31, 2026.

The decision moment is the beginning of March 25, so the model features are created before the outcome window starts.

### 4. What I predict or rank

I rank content items for possible content-refresh review.

My decline proxy equals 1 when impressions during March 25–31 are less than 80% of impressions during March 18–24.

I require at least 10 baseline impressions to avoid creating labels from extremely small numbers.

### 5. What I deliberately exclude

I deliberately exclude future outcome impressions, the future-to-baseline ratio, and the decline label from the final feature set.

These fields contain information that is unavailable at the decision moment and would cause data leakage.


In [1]:
%pip -q install -U duckdb huggingface_hub pandas scikit-learn

from google.colab import userdata
from huggingface_hub import snapshot_download

import glob
import duckdb
import numpy as np
import pandas as pd

# Read the token securely from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Open the key icon in Colab, "
        "add HF_TOKEN, and enable Notebook access."
    )

print("Downloading only the March 2026 warehouse partition...")

warehouse_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir="/content/flyrank_warehouse",
    allow_patterns=[
        "fact_content_daily_performance/month=2026-03/*.parquet"
    ],
)

march_pattern = (
    "/content/flyrank_warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

march_files = glob.glob(march_pattern)

if not march_files:
    raise FileNotFoundError(
        "No March Parquet files were downloaded. "
        "Check your Hugging Face dataset access and token."
    )

print("March files found:", len(march_files))

con = duckdb.connect()

DAILY_MARCH = (
    f"read_parquet("
    f"'{march_pattern}', "
    f"hive_partitioning=true)"
)

print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
test_result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {DAILY_MARCH}
""").df()

test_result

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## 2. Fields: feature / label / context / excluded

### Features

* `impressions_pre24`
* `clicks_pre24`
* `ctr_pre24`
* `avg_position_pre24`
* `active_days_pre24`

These five fields use only information available on or before March 24, 2026.

### Label and label-building fields

* `baseline_impressions_7d`
* `outcome_impressions_7d`
* `is_declining_proxy`

The label is 1 when outcome impressions are less than 80% of baseline impressions.

### Context fields

* `client_hash_id`
* `content_hash_id`

These fields identify and group rows. They are not given to the model as features.

### Excluded fields

* Future outcome impressions
* Future-to-baseline ratio
* `is_declining_proxy` as an input feature
* Client and content IDs as model features
* June 2026 data
* The `_sample` table

Future information would create leakage. IDs are context rather than behavioral measurements. June is kept sealed.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FEATURES = [
    "impressions_pre24",
    "clicks_pre24",
    "ctr_pre24",
    "avg_position_pre24",
    "active_days_pre24",
]

CONTEXT = [
    "client_hash_id",
    "content_hash_id",
]

LABEL = "is_declining_proxy"

print("Number of planned features:", len(FEATURES))
print("Features:", FEATURES)

Number of planned features: 5
Features: ['impressions_pre24', 'clicks_pre24', 'ctr_pre24', 'avg_position_pre24', 'active_days_pre24']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification Query 1: prove the raw daily grain

Q1_GRAIN = f"""
WITH key_counts AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM {DAILY_MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
)

SELECT
    COUNT(*) AS distinct_daily_keys,

    SUM(
        CASE
            WHEN rows_per_key > 1 THEN 1
            ELSE 0
        END
    ) AS duplicate_key_groups,

    MAX(rows_per_key) AS maximum_rows_per_key

FROM key_counts
"""

grain_result = con.sql(Q1_GRAIN).df()
grain_result


NameError: name 'DAILY_MARCH' is not defined

In [5]:
# Verification Query 2: row count and date span

Q2_COUNT_AND_DATES = f"""
SELECT
    COUNT(*) AS total_daily_rows,
    COUNT(DISTINCT client_hash_id) AS number_of_clients,
    COUNT(DISTINCT content_hash_id) AS number_of_content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {DAILY_MARCH}
"""

count_result = con.sql(Q2_COUNT_AND_DATES).df()
count_result

NameError: name 'DAILY_MARCH' is not defined

In [6]:
# Verification Query 3: availability using IS TRUE

Q3_AVAILABILITY = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_rows_surviving,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_rows_surviving,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_rows_surviving,

    ROUND(
        100.0 *
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) /
        NULLIF(COUNT(*), 0),
        2
    ) AS gsc_survival_percentage

FROM {DAILY_MARCH}
"""

availability_result = con.sql(Q3_AVAILABILITY).df()
availability_result

NameError: name 'DAILY_MARCH' is not defined

In [7]:
# Build one modeling row per client-content pair.
# This is the feature-building cell, not a fourth verification query.

FEATURE_FRAME_SQL = f"""
WITH per_content AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01' AND DATE '2026-03-24'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS impressions_pre24,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01' AND DATE '2026-03-24'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS clicks_pre24,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01' AND DATE '2026-03-24'
                     AND gsc_impressions > 0
                     AND gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) AS weighted_position_total,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01' AND DATE '2026-03-24'
                     AND gsc_impressions > 0
                     AND gsc_avg_position > 0
                THEN gsc_impressions
                ELSE 0
            END
        ) AS positioned_impressions,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-01' AND DATE '2026-03-24'
                     AND gsc_impressions > 0
                THEN report_date
            END
        ) AS active_days_pre24,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-18' AND DATE '2026-03-24'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS baseline_impressions_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-25' AND DATE '2026-03-31'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions_7d,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-18' AND DATE '2026-03-24'
                THEN report_date
            END
        ) AS baseline_available_days,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN
                     DATE '2026-03-25' AND DATE '2026-03-31'
                THEN report_date
            END
        ) AS outcome_available_days

    FROM {DAILY_MARCH}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

finished AS (
    SELECT
        client_hash_id,
        content_hash_id,

        impressions_pre24,
        clicks_pre24,

        100.0 * clicks_pre24 /
        NULLIF(impressions_pre24, 0) AS ctr_pre24,

        weighted_position_total /
        NULLIF(positioned_impressions, 0) AS avg_position_pre24,

        active_days_pre24,

        baseline_impressions_7d,
        outcome_impressions_7d,

        CASE
            WHEN outcome_impressions_7d
                 < 0.80 * baseline_impressions_7d
            THEN 1
            ELSE 0
        END AS is_declining_proxy

    FROM per_content

    WHERE impressions_pre24 > 0
      AND baseline_impressions_7d >= 10
      AND baseline_available_days = 7
      AND outcome_available_days = 7
)

SELECT *
FROM finished
ORDER BY
    client_hash_id,
    content_hash_id
"""

model_frame = con.sql(FEATURE_FRAME_SQL).df()

feature_frame = model_frame[
    CONTEXT + FEATURES + [LABEL]
].copy()

print("Modeling rows:", len(feature_frame))
print("Clients:", feature_frame["client_hash_id"].nunique())
print("Number of features:", len(FEATURES))
print("Decline rate:", feature_frame[LABEL].mean())

feature_frame.head(10)

NameError: name 'DAILY_MARCH' is not defined

### Why each feature is available at the decision moment

1. **impressions_pre24** — knowable at the decision moment because it only sums impressions observed from March 1 through March 24.

2. **clicks_pre24** — knowable at the decision moment because all included clicks occurred on or before March 24.

3. **ctr_pre24** — knowable at the decision moment because it is calculated only from pre-decision clicks and impressions.

4. **avg_position_pre24** — knowable at the decision moment because it uses positions and impressions observed on or before March 24.

5. **active_days_pre24** — knowable at the decision moment because it only counts pre-decision dates with observed impressions.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

X = feature_frame[FEATURES].replace(
    [np.inf, -np.inf],
    np.nan
)

y = feature_frame[LABEL].astype(int)

print("Label counts:")
print(y.value_counts())

if y.nunique() != 2:
    raise ValueError(
        "The label does not contain two classes. "
        "Check the feature-building query."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "tree",
        DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=42
        )
    )
])

honest_model.fit(X_train, y_train)

honest_probabilities = honest_model.predict_proba(
    X_test
)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    honest_probabilities
)

print(f"Honest ROC AUC: {honest_auc:.3f}")

NameError: name 'feature_frame' is not defined

In [ ]:
# Intentionally create one future-derived leaking feature.

feature_frame["future_outcome_ratio"] = (
    model_frame["outcome_impressions_7d"] /
    model_frame["baseline_impressions_7d"]
)

LEAKY_FEATURES = FEATURES + [
    "future_outcome_ratio"
]

X_leaky = feature_frame[LEAKY_FEATURES].replace(
    [np.inf, -np.inf],
    np.nan
)

X_leaky_train = X_leaky.iloc[X_train.index]
X_leaky_test = X_leaky.iloc[X_test.index]

leaky_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "tree",
        DecisionTreeClassifier(
            max_depth=1,
            random_state=42
        )
    )
])

leaky_model.fit(
    X_leaky_train,
    y_train
)

leaky_probabilities = leaky_model.predict_proba(
    X_leaky_test
)[:, 1]

leaky_auc = roc_auc_score(
    y_test,
    leaky_probabilities
)

print(f"Honest ROC AUC: {honest_auc:.3f}")
print(f"Leaky ROC AUC:  {leaky_auc:.3f}")

In [9]:
feature_frame.drop(
    columns=["future_outcome_ratio"],
    inplace=True
)

assert "future_outcome_ratio" not in feature_frame.columns
assert len(FEATURES) == 5

print("Leakage feature removed.")
print("Final features:", FEATURES)
print(f"Final score kept: Honest ROC AUC = {honest_auc:.3f}")

NameError: name 'feature_frame' is not defined

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

### Named limitation — short-window noise

This analysis uses only one month and a seven-day future outcome window.

A campaign, holiday, reporting interruption, normal weekly variation, or temporary ranking change could appear as content decline.

Therefore, this proxy can support a human-review queue, but it cannot prove permanent content decay or prove that refreshing a page will cause performance to improve.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
assert "future_outcome_ratio" not in feature_frame.columns
assert len(FEATURES) == 5

print("Final feature frame is leakage-free.")
print("Final feature count:", len(FEATURES))

NameError: name 'feature_frame' is not defined

## 5. Self-check

- [x] Five plain-word contract answers are present
- [x] Exactly three verification queries have visible outputs
- [x] Availability was checked using `IS TRUE`
- [x] The feature frame has exactly five features
- [x] Every feature has an “available when?” explanation
- [x] One future label-derived feature was deliberately added
- [x] The leaky score was displayed
- [x] The leaking feature was deleted
- [x] The honest score was kept
- [x] One named limitation is included
- [x] June 2026 and the `_sample` table were not used
- [x] No token, client name, URL, or private query is displayed